In [2]:
# conda activate anndata

import os
import re
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
from scipy import sparse
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

pd.set_option('display.max_columns', None)

In [3]:
psi = pd.read_csv("data/yao_2021_ACA_MOp_VISp_STAR_SyntheticDataset1_10pcntCells_20SD_200samples_SJ_PSI.csv", index_col=0)

In [4]:
# Load single-cell PSI data
sdata = ad.read_h5ad("data/yao_2021_ACA_MOp_VISp_STAR_SJ_counts_PSI_annotated.h5ad")

# Load single-cell gene expression data
adata = ad.read_h5ad("data/yao_2021_ACA_MOp_VISp_STAR_model/yao_2021_ACA_MOp_VISp_STAR_gene_counts_scVI.h5ad")

adata.obs['subclass_label'] = adata.obs['subclass_label'].astype(str).str.replace("/", "_", regex=False).str.replace(" ", "_", regex=False)
sdata.obs['subclass_label'] = sdata.obs['subclass_label'].astype(str).str.replace("/", "_", regex=False).str.replace(" ", "_", regex=False)

# Work with full gene space
adata_raw = adata.raw.to_adata()
adata_raw.X = adata_raw.X.toarray()

In [5]:
adata.obs.value_counts('subclass_label')

subclass_label
L4_5_IT_CTX        3544
L6_CT_CTX          3425
Vip                2446
L6_IT_CTX          2353
Pvalb              1820
L2_3_IT_CTX        1779
Sst                1725
L5_IT_CTX          1579
Lamp5              1396
L6b_CTX            1169
L5_6_NP_CTX         882
L5_PT_CTX           498
Sncg                331
Astro               215
L2_3_IT_PPP         143
Sst_Chodl           100
Oligo                97
Endo                 69
SMC-Peri             49
Micro-PVM            46
VLMC                 42
Car3                 42
Meis2                38
L5_6_IT_TPE-ENT       8
CR                    4
L4_RSP-ACA            1
Name: count, dtype: int64

### Pairwise DE genes

In [13]:
# def _col_idx_for(df, stat):
#     return df.columns.str.contains(stat)

def _safe(name):
    # simple filename sanitizer
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', str(name))

def _barplot_colors(ctypes, working_ctype):
    colors = ["tab:blue"] * len(ctypes)
    idx = pd.Index(ctypes).get_indexer([working_ctype])[0]
    colors[idx] = "tab:red"
    return colors

def _flatten_1d(x):
    if sparse.issparse(x):   # AnnData/sparse safe
        return x.toarray().ravel()
    return np.asarray(x, dtype=float).ravel()

def plot_barplot_by_cell_type(means, errs, ctypes, working_ctype, title, ylabel, show=True):
    means = np.asarray(means)
    errs = np.asarray(errs) * 2.0 
    
    colors = _barplot_colors(ctypes, working_ctype)
    x = np.arange(len(ctypes))
    fig, ax = plt.subplots()
    ax.bar(x, means, yerr=errs, capsize=3, color=colors)
    ax.set_xticks(x); ax.set_xticklabels(ctypes, rotation=45, ha="right", fontsize=8)
    ax.set_ylabel(ylabel); ax.set_title(title)
    fig.tight_layout()
    if show:
        plt.show()
    return fig

def plot_ME_vs_PSI(mod_eig, exon_psi, title, show=True):
    fig, ax = plt.subplots()
    ax.scatter(mod_eig, exon_psi, s=18)
    ax.set_xlabel("Module eigengene")
    ax.set_ylabel("Exon PSI")
    ax.set_title(title)
    plt.tight_layout()
    if show:
        plt.show()
    return fig

def violin_with_points(values_by_ct, ctypes, focus=None,
                       base_fc='tab:blue', highlight_fc='tab:red',
                       ylabel="Value", title=None,
                       jitter=0.08, point_size=8, point_alpha=0.35,
                       max_points_per_ct=None, seed=0, violin_alpha=0.5,
                       show=False):                         # <- default False when saving
    rng = np.random.default_rng(seed)
    pos = np.arange(1, len(ctypes)+1)

    fig, ax = plt.subplots()                                # <- create fig
    vp = ax.violinplot(values_by_ct, positions=pos, showmeans=True, showextrema=True)

    # color violins (no outlines)
    for i, b in enumerate(vp['bodies']):
        col = highlight_fc if (focus is not None and ctypes[i] == focus) else base_fc
        b.set_facecolor(col)
        b.set_edgecolor('none')
        b.set_alpha(violin_alpha)
    for k in ('cmeans','cmins','cmaxes','cbars','cmedians'):
        if k in vp: vp[k].set_visible(False)

    # jitter points
    for i, v in enumerate(values_by_ct, start=1):
        v = np.asarray(v, float)
        v = v[np.isfinite(v)]
        if v.size == 0: continue
        if max_points_per_ct and v.size > max_points_per_ct:
            v = v[rng.choice(v.size, size=max_points_per_ct, replace=False)]
        x = i + (rng.random(v.size) - 0.5) * 2 * jitter
        col = highlight_fc if (focus is not None and ctypes[i-1] == focus) else base_fc
        ax.scatter(x, v, s=point_size, alpha=point_alpha, color=col, edgecolors='none', rasterized=True)

    ax.set_xticks(pos); ax.set_xticklabels(ctypes, rotation=45, ha='right', fontsize=8)
    ax.set_ylabel(ylabel); 
    if title: ax.set_title(title)
    fig.tight_layout()

    if show:
        plt.show()

    return fig  

In [14]:
data_source = f"yao_2021_ACA_MOp_VISp_STAR_SyntheticDataset1_10pcntCells_20SD_200samples_mergeParam0.95_subsetCutoff825.95_Modules_yao_2021_ACA_MOp_VISp_STAR_donor_subclass_label_pseudobulk_pairwise_DE_genes_strictFALSE_logFC3_nSignif21_enrichments"

psi = pd.read_csv("data/yao_2021_ACA_MOp_VISp_STAR_SyntheticDataset1_10pcntCells_20SD_200samples_SJ_PSI.csv", index_col=0)
corr_df = pd.read_csv(f"data/corrs/yao_2021_ACA_MOp_VISp_STAR_SyntheticDataset1_10pcntCells_20SD_200samples_mergeParam0.95_subsetCutoff825.95_Modules_yao_2021_ACA_MOp_VISp_STAR_donor_subclass_label_pseudobulk_pairwise_DE_genes_strictFALSE_logFC3_nSignif21_enrichments_PSI_vs_exon_corr.csv", index_col=0)
top_qval_mods_df = pd.read_csv("data/enrichments/yao_2021_ACA_MOp_VISp_STAR_SyntheticDataset1_10pcntCells_20SD_200samples_mergeParam0.95_subsetCutoff825.95_Modules_yao_2021_ACA_MOp_VISp_STAR_donor_subclass_label_pseudobulk_pairwise_DE_genes_strictFALSE_logFC3_nSignif21_enrichments_abund_corr.csv")

In [15]:
# Remove genes correlated with abundance < .75

mask = ~(top_qval_mods_df['Old_cor'] < .75)
top_qval_mods_df = top_qval_mods_df.loc[mask]

In [16]:
top_qval_mods_df.index = top_qval_mods_df['Cell_type']
gene_exon_df = corr_df['Gene']

In [17]:
sc_ctypes = np.unique(adata.obs['subclass_label'])
sc_ctypes 

array(['Astro', 'CR', 'Car3', 'Endo', 'L2_3_IT_CTX', 'L2_3_IT_PPP',
       'L4_5_IT_CTX', 'L4_RSP-ACA', 'L5_6_IT_TPE-ENT', 'L5_6_NP_CTX',
       'L5_IT_CTX', 'L5_PT_CTX', 'L6_CT_CTX', 'L6_IT_CTX', 'L6b_CTX',
       'Lamp5', 'Meis2', 'Micro-PVM', 'Oligo', 'Pvalb', 'SMC-Peri',
       'Sncg', 'Sst', 'Sst_Chodl', 'VLMC', 'Vip'], dtype=object)

In [18]:
ctypes = corr_df.columns[1:]

In [21]:
top_n = 15
ascending = True

for w_ctype in ctypes:
    print(w_ctype)
    outdir = f"figures/pseudobulk/{data_source}/{w_ctype}"
    print(outdir)
        
    mask = top_qval_mods_df['Cell_type'] == w_ctype 
    row = top_qval_mods_df[mask]
    mod_df = pd.read_csv(row['Old_ME_path'].item())
    mod_eig_df = mod_df.set_index("Sample")[row['Old_module'].item()]
    mod_eig = pd.to_numeric(mod_eig_df, errors="coerce")

    # Get mean expression of cell type exon/gene in each cell type

    corr_df = corr_df.sort_values(w_ctype, ascending=ascending)
    top_exons = corr_df[w_ctype].index.tolist()[:top_n]

    cols = [f"{ct}_{stat}" for ct in ctypes for stat in ("mean", "SE")]
    ctype_psi_df = pd.DataFrame(columns=cols, index=top_exons) 
    ctype_expr_df = pd.DataFrame(columns=cols, index=top_exons)

    for exon in top_exons:
        exon_mask = sdata.var_names.isin([exon])
        sdata_sub = sdata[:, exon_mask].copy()
        gene_mask = adata_raw.var_names.isin([gene_exon_df.loc[exon]])
        adata_sub = adata_raw[:, gene_mask].copy()
        
        psi_vals_by_ct = []
        expr_vals_by_ct = []
        mean_psi_by_ct = [] 
        mean_psi_se_by_ct = []
        mean_expr_by_ct = []
        mean_expr_se_by_ct = []

        for ct in sc_ctypes:
            cell_mask = adata_sub.obs['subclass_label'] == ct
            n = np.sum(cell_mask)

            psi_per_cell = _flatten_1d(sdata_sub.X[cell_mask, :])
            expr_per_cell = _flatten_1d(adata_sub.X[cell_mask, :])

            psi_vals_by_ct.append(psi_per_cell)
            expr_vals_by_ct.append(np.log1p(expr_per_cell))

            mean_psi_by_ct.append(np.mean(psi_per_cell))
            mean_psi_se_by_ct.append(np.sqrt(np.var(psi_per_cell) / n))

            mean_expr = np.mean(adata_raw.X[cell_mask, :]) 
            mean_expr_by_ct.append(np.mean(expr_per_cell))
            mean_expr_se_by_ct.append(np.sqrt(np.var(expr_per_cell) / n) / mean_expr)

        corr = round(corr_df.loc[exon, w_ctype], 2)
        gene = gene_exon_df.loc[exon]
        exon_psi = psi.loc[exon]
        exon_label = ''.join(str(exon).split("_")[1:])
        pdf_path = f"{outdir}/{_safe(w_ctype)}_{_safe(gene)}_{_safe(exon_label)}_ascending{ascending}.pdf"

        with PdfPages(pdf_path) as pdf:
            fig = plot_barplot_by_cell_type(mean_psi_by_ct, mean_psi_se_by_ct, 
                                            sc_ctypes, w_ctype,
                                            title=f"PSI for {gene} {exon_label} exon",
                                            ylabel="Mean PSI", show=False)
            pdf.savefig(fig); plt.close(fig)

            fig = plot_barplot_by_cell_type(mean_expr_by_ct, mean_expr_se_by_ct, 
                                            sc_ctypes, w_ctype,
                                            title=f"Gene expression for {gene}",
                                            ylabel="Mean expression (normalized)", 
                                            show=False)
            pdf.savefig(fig); plt.close(fig)

            fig = plot_ME_vs_PSI(mod_eig, exon_psi,
                                 title=f"{w_ctype} ME vs. PSI for {gene} {exon_label} exon\nCorr: {corr}",
                                 show=False)
            pdf.savefig(fig); plt.close(fig)

            fig = violin_with_points(psi_vals_by_ct, sc_ctypes, focus=w_ctype,
                                     ylabel="PSI",
                                     title=f"PSI distribution for {gene} {exon_label} exon",
                                     jitter=0.2, max_points_per_ct=3000, violin_alpha=0.2, 
                                     show=False)
            pdf.savefig(fig); plt.close(fig)
            
            fig = violin_with_points(expr_vals_by_ct, sc_ctypes, focus=w_ctype,
                                     ylabel="Expression (log2)",
                                     title=f"Count distribution for {gene}",
                                     jitter=0.2, max_points_per_ct=3000, violin_alpha=0.2, 
                                     show=False)
            pdf.savefig(fig); plt.close(fig)

        print("Saved", exon)
        print("")
    

Pvalb
figures/pseudobulk/yao_2021_ACA_MOp_VISp_STAR_SyntheticDataset1_10pcntCells_20SD_200samples_mergeParam0.95_subsetCutoff825.95_Modules_yao_2021_ACA_MOp_VISp_STAR_donor_subclass_label_pseudobulk_pairwise_DE_genes_strictFALSE_logFC3_nSignif21_enrichments/Pvalb
Saved ENSMUSG00000033419_ProteinCoding_4

Saved ENSMUSG00000052889_NMD_1

Saved ENSMUSG00000028478_ProteinCoding_4

Saved ENSMUSG00000022307_ProteinCoding_1

Saved ENSMUSG00000026179_ProteinCoding_1

Saved ENSMUSG00000035681_ProteinCoding_1

Saved ENSMUSG00000023089_ProteinCoding_1

Saved ENSMUSG00000062785_ProteinCoding_2

Saved ENSMUSG00000025221_other_1

Saved ENSMUSG00000032336_ProteinCoding_1

Saved ENSMUSG00000033287_ProteinCoding_3

Saved ENSMUSG00000033287_NMD_1

Saved ENSMUSG00000029095_ProteinCoding_3

Saved ENSMUSG00000015804_ProteinCoding_2

Saved ENSMUSG00000022378_ProteinCoding_1

L2_3_IT_CTX
figures/pseudobulk/yao_2021_ACA_MOp_VISp_STAR_SyntheticDataset1_10pcntCells_20SD_200samples_mergeParam0.95_subsetCutoff825

In [22]:
ct

'Micro-PVM'

In [23]:
ctypes

Index(['Pvalb', 'L2_3_IT_CTX', 'L6b_CTX', 'Lamp5', 'L5_PT_CTX', 'Sncg',
       'L5_6_NP_CTX', 'Sst_Chodl', 'Meis2', 'Oligo', 'Car3', 'VLMC',
       'SMC-Peri', 'Astro', 'Endo', 'Micro-PVM'],
      dtype='object')

In [24]:
psi_vals_by_ct

[array([0., 0., 0., ..., 0., 0., 0.]),
 array([0., 0., 0., ..., 0., 0., 0.]),
 array([0., 0., 0., ..., 0., 0., 0.]),
 array([0., 0., 0., ..., 0., 0., 0.]),
 array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        

### Claude genes

In [22]:
# def _col_idx_for(df, stat):
#     return df.columns.str.contains(stat)

def _safe(name):
    # simple filename sanitizer
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', str(name))

def _barplot_colors(ctypes, working_ctype):
    colors = ["tab:blue"] * len(ctypes)
    idx = pd.Index(ctypes).get_indexer([working_ctype])[0]
    colors[idx] = "tab:red"
    return colors

def _flatten_1d(x):
    if sparse.issparse(x):   # AnnData/sparse safe
        return x.toarray().ravel()
    return np.asarray(x, dtype=float).ravel()

def plot_barplot_by_cell_type(means, errs, ctypes, title, ylabel, show=True):
    # mean_cols = _col_idx_for(df, "mean")
    # se_cols  = _col_idx_for(df, "SE")
    # means = df.loc[exon, mean_cols].astype(float).values
    # errs = df.loc[exon, se_cols].astype(float).values * 2.0
   
    means = np.asarray(means)
    errs = np.asarray(errs) * 2.0 
    
    # colors = _barplot_colors(ctypes, working_ctype)
    x = np.arange(len(ctypes))
    fig, ax = plt.subplots()
    ax.bar(x, means, yerr=errs, capsize=3) # color=colors)
    ax.set_xticks(x); ax.set_xticklabels(ctypes, rotation=45, ha="right", fontsize=8)
    ax.set_ylabel(ylabel); ax.set_title(title)
    fig.tight_layout()
    if show:
        plt.show()
    return fig


def plot_ME_vs_PSI(mod_eig, exon_psi, title, show=True):
    fig, ax = plt.subplots()
    ax.scatter(mod_eig, exon_psi, s=18)
    ax.set_xlabel("Module eigengene")
    ax.set_ylabel("Exon PSI")
    ax.set_title(title)
    plt.tight_layout()
    if show:
        plt.show()
    return fig

def violin_with_points(values_by_ct, ctypes, focus=None,
                       base_fc='tab:blue', highlight_fc='tab:red',
                       ylabel="Value", title=None,
                       jitter=0.08, point_size=8, point_alpha=0.35,
                       max_points_per_ct=None, seed=0, violin_alpha=0.5,
                       show=False):                         # <- default False when saving
    rng = np.random.default_rng(seed)
    pos = np.arange(1, len(ctypes)+1)

    fig, ax = plt.subplots()                                # <- create fig
    vp = ax.violinplot(values_by_ct, positions=pos, showmeans=True, showextrema=True)

    # color violins (no outlines)
    for i, b in enumerate(vp['bodies']):
        col = highlight_fc if (focus is not None and ctypes[i] == focus) else base_fc
        b.set_facecolor(col)
        b.set_edgecolor('none')
        b.set_alpha(violin_alpha)
    for k in ('cmeans','cmins','cmaxes','cbars','cmedians'):
        if k in vp: vp[k].set_visible(False)

    # jitter points
    for i, v in enumerate(values_by_ct, start=1):
        v = np.asarray(v, float)
        v = v[np.isfinite(v)]
        if v.size == 0: continue
        if max_points_per_ct and v.size > max_points_per_ct:
            v = v[rng.choice(v.size, size=max_points_per_ct, replace=False)]
        x = i + (rng.random(v.size) - 0.5) * 2 * jitter
        col = highlight_fc if (focus is not None and ctypes[i-1] == focus) else base_fc
        ax.scatter(x, v, s=point_size, alpha=point_alpha, color=col, edgecolors='none', rasterized=True)

    ax.set_xticks(pos); ax.set_xticklabels(ctypes, rotation=45, ha='right', fontsize=8)
    ax.set_ylabel(ylabel); 
    if title: ax.set_title(title)
    fig.tight_layout()

    if show:
        plt.show()

    return fig  

In [70]:
data_source = f"yao_2021_ACA_MOp_VISp_STAR_SyntheticDataset1_10pcntCells_20SD_200samples_mergeParam0.95_subsetCutoff825.95_Modules_Claude_marker_genes_enrichments_ctype_abundance_PSI_vs_exon_corr.csv"
corr_df = pd.read_csv(f"data/corrs/yao_2021_ACA_MOp_VISp_STAR_SyntheticDataset1_10pcntCells_20SD_200samples_mergeParam0.95_subsetCutoff825.95_Modules_Claude_marker_genes_enrichments_ctype_abundance_PSI_vs_exon_corr.csv", index_col=0)
mod_eig_df = pd.read_csv("data/yao_2021_ACA_MOp_VISp_STAR_SyntheticDataset1_10pcntCells_20SD_200samples_mergeParam0.95_subsetCutoff825.95_Modules_Claude_marker_genes_enrichments_ctype_abundance.csv", index_col=0)

In [71]:
mod_eig_df.head()

,All_GABAergic,All_Glutamatergic,All_Neuronal,Sst
Sample,,,,
Sample1,-0.084202,0.144404,0.039640,0.027145
Sample10,0.016585,0.053104,0.077453,-0.071263
Sample100,0.191336,-0.051171,0.026040,0.150213
Sample101,0.028227,-0.013794,-0.015006,-0.021221
Sample102,0.116253,-0.022771,0.146024,0.122723


In [72]:
gene_exon_df = corr_df['Gene']

In [73]:
ctypes = mod_eig_df.columns

In [ ]:
# sc_ctypes = np.unique(adata.obs['subclass_label'])
# sc_ctypes 

In [74]:
top_qval_mods_df = pd.read_csv("data/enrichments/yao_2021_ACA_MOp_VISp_STAR_SyntheticDataset1_10pcntCells_20SD_200samples_mergeParam0.95_subsetCutoff825.95_Modules_yao_2021_ACA_MOp_VISp_STAR_donor_subclass_label_pseudobulk_pairwise_DE_genes_strictFALSE_logFC3_nSignif21_enrichments_abund_corr.csv")
orig_ctypes = top_qval_mods_df['Cell_type']
orig_ctypes = np.sort(orig_ctypes)
orig_ctypes 

array(['Astro', 'Car3', 'Endo', 'L2_3_IT_CTX', 'L4_5_IT_CTX',
       'L5_6_NP_CTX', 'L5_PT_CTX', 'L6_CT_CTX', 'L6_IT_CTX', 'L6b_CTX',
       'Lamp5', 'Meis2', 'Micro-PVM', 'Oligo', 'Pvalb', 'SMC-Peri',
       'Sncg', 'Sst', 'Sst_Chodl', 'VLMC', 'Vip'], dtype=object)

In [77]:
top_n = 15
ascending = True 

for w_ctype in ctypes:
    print(w_ctype)
    outdir = f"figures/{data_source}/{w_ctype}"
    os.makedirs(outdir, exist_ok=True)
    
    mod_eig = pd.to_numeric(mod_eig_df[w_ctype], errors="coerce") 
    mod_eig = mod_eig.reindex(psi.columns) 
    # Get mean expression of cell type exon/gene in each cell type

    corr_df = corr_df.sort_values(w_ctype, ascending=ascending)
    top_exons = corr_df[w_ctype].index.tolist()[:top_n]

    cols = [f"{ct}_{stat}" for ct in ctypes for stat in ("mean", "SE")]
    ctype_psi_df = pd.DataFrame(columns=cols, index=top_exons) 
    ctype_expr_df = pd.DataFrame(columns=cols, index=top_exons)

    for exon in top_exons:
        exon_mask = sdata.var_names.isin([exon])
        sdata_sub = sdata[:, exon_mask].copy()
        gene_mask = adata_raw.var_names.isin([gene_exon_df.loc[exon]])
        adata_sub = adata_raw[:, gene_mask].copy()
        
        psi_vals_by_ct = []
        expr_vals_by_ct = []
        mean_psi_by_ct = [] 
        mean_psi_se_by_ct = []
        mean_expr_by_ct = []
        mean_expr_se_by_ct = []

        for ct in orig_ctypes:
            cell_mask = adata_sub.obs['subclass_label'] == ct

            psi_per_cell = _flatten_1d(sdata_sub.X[cell_mask, :])
            expr_per_cell = _flatten_1d(adata_sub.X[cell_mask, :])

            psi_vals_by_ct.append(psi_per_cell)
            expr_vals_by_ct.append(np.log1p(expr_per_cell))

            mean_psi_by_ct.append(np.mean(psi_per_cell))
            mean_psi_se_by_ct.append(np.sqrt(np.var(psi_per_cell) / n))

            mean_expr = np.mean(adata_raw.X[cell_mask, :]) 
            mean_expr_by_ct.append(np.mean(expr_per_cell))
            mean_expr_se_by_ct.append(np.sqrt(np.var(expr_per_cell) / n) / mean_expr)

        corr = round(corr_df.loc[exon, w_ctype], 2)
        gene = gene_exon_df.loc[exon]
        exon_psi = psi.loc[exon]
        exon_label = ''.join(str(exon).split("_")[1:])
        pdf_path = f"{outdir}/{_safe(w_ctype)}_{_safe(gene)}_{_safe(exon_label)}_ascending{ascending}.pdf"

        with PdfPages(pdf_path) as pdf:
            fig = plot_barplot_by_cell_type(mean_psi_by_ct, mean_psi_se_by_ct, 
                                            orig_ctypes, 
                                            title=f"PSI for {gene} {exon_label} exon",
                                            ylabel="Mean PSI", show=False)
            pdf.savefig(fig); plt.close(fig)

            fig = plot_barplot_by_cell_type(mean_expr_by_ct, mean_expr_se_by_ct, 
                                            orig_ctypes, 
                                            title=f"Gene expression for {gene}",
                                            ylabel="Mean expression (normalized)", 
                                            show=False)
            pdf.savefig(fig); plt.close(fig)

            fig = plot_ME_vs_PSI(mod_eig, exon_psi,
                                 title=f"{w_ctype} ME vs. PSI for {gene} {exon_label} exon\nCorr: {corr}",
                                 show=False)
            pdf.savefig(fig); plt.close(fig)

 
            fig = violin_with_points(psi_vals_by_ct, orig_ctypes, focus=None,
                                     ylabel="PSI",
                                     title=f"PSI distribution for {gene} {exon_label} exon",
                                     jitter=0.2, max_points_per_ct=3000, violin_alpha=0.2, 
                                     show=False)
            pdf.savefig(fig); plt.close(fig)
            
            fig = violin_with_points(expr_vals_by_ct, orig_ctypes, focus=None,
                                     ylabel="Expression (log2)",
                                     title=f"Count distribution for {gene}",
                                     jitter=0.2, max_points_per_ct=3000, violin_alpha=0.2, 
                                     show=False)
            pdf.savefig(fig); plt.close(fig)

        print("Saved", exon)
        print("")
    

All_GABAergic
Saved ENSMUSG00000036564_ProteinCoding_2

Saved ENSMUSG00000032336_ProteinCoding_1

Saved ENSMUSG00000015759_ProteinCoding_1

Saved ENSMUSG00000052889_NMD_1

Saved ENSMUSG00000021546_ProteinCoding_1

Saved ENSMUSG00000028478_ProteinCoding_4

Saved ENSMUSG00000031708_ProteinCoding_1

Saved ENSMUSG00000023089_ProteinCoding_1

Saved ENSMUSG00000034675_ProteinCoding_1

Saved ENSMUSG00000058589_ProteinCoding_9

Saved ENSMUSG00000058589_ProteinCoding_10

Saved ENSMUSG00000033419_ProteinCoding_4

Saved ENSMUSG00000086316_ProteinCoding_2

Saved ENSMUSG00000022307_ProteinCoding_1

Saved ENSMUSG00000003279_ProteinCoding_3

All_Glutamatergic
Saved ENSMUSG00000025871_ProteinCoding_4

Saved ENSMUSG00000025871_ProteinCoding_5

Saved ENSMUSG00000006418_ProteinCoding_1

Saved ENSMUSG00000030805_ProteinCoding_2

Saved ENSMUSG00000090841_ProteinCoding_3

Saved ENSMUSG00000025871_ProteinCoding_7

Saved ENSMUSG00000040242_ProteinCoding_2

Saved ENSMUSG00000040407_ProteinCoding_1

Saved ENSMU